# 30 · Conditional Signal Decomposition

This notebook localizes the zeta–GUE overlap phase by testing which **conditional window regimes** retain the decomposition signal.

Previous notebooks showed:
- zeta vs GUE has a stable overlap phase
- the phase survives several perturbations
- histogram-matched and block-preserving surrogates preserve much of the phase
- global smoothing tends to destroy it

Now we ask:

> In which local regimes does that phase live most strongly?

## Conditions

We condition on local window statistics from the same feature pipeline:
- **low entropy**
- **high entropy**
- **low unevenness**
- **high unevenness**

## Decomposition families

Within each condition, we compare:
- **baseline**
- **distribution_only**
- **iid_histogram**
- **local_blocks**
- **order_only**
- **global_trend**

## Main outputs

- conditional phase-gap bars
- conditional phase-gap heatmaps
- condition-difference heatmaps
- bootstrap error-bar comparisons for selected surrogates

In [ ]:
import numpy as np
import mpmath as mp
import matplotlib.pyplot as plt

mp.mp.dps = 50
rng = np.random.default_rng(9423)

## Base data

In [ ]:
N = 2200
zeros = [mp.zetazero(n) for n in range(1, N + 1)]
t = np.array([float(mp.im(z)) for z in zeros])
zeta_gaps_full = np.diff(t)

poisson_gaps_full = rng.exponential(scale=1.0, size=len(zeta_gaps_full))

def sample_gue_bulk_spacings(matrix_size=140, n_matrices=70, bulk_fraction=0.5, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    all_gaps = []
    for _ in range(n_matrices):
        real = rng.normal(size=(matrix_size, matrix_size))
        imag = rng.normal(size=(matrix_size, matrix_size))
        A = real + 1j * imag
        H = (A + A.conj().T) / 2.0
        eigvals = np.linalg.eigvalsh(H)
        eigvals = np.sort(eigvals)

        n = len(eigvals)
        keep = int(n * bulk_fraction)
        start = (n - keep) // 2
        stop = start + keep
        bulk_vals = eigvals[start:stop]
        all_gaps.extend(np.diff(bulk_vals).tolist())

    return np.array(all_gaps)

gue_gaps_full = sample_gue_bulk_spacings(matrix_size=140, n_matrices=70, bulk_fraction=0.5, rng=rng)

len(zeta_gaps_full), len(poisson_gaps_full), len(gue_gaps_full)

## Window statistics and features

In [ ]:
def extract_windows(gaps, k=5):
    return np.array([gaps[i:i+k] for i in range(len(gaps) - k + 1)])

def normalize_window(window):
    w = np.array(window, dtype=float)
    return w / w.mean()

def unevenness(window):
    return float(np.max(window) - np.min(window))

def reversal_symmetry_score(window):
    return float(np.mean(np.abs(window - window[::-1])))

def window_entropy(window):
    eps = 1e-12
    p = window / np.sum(window)
    return float(-np.sum(p * np.log(p + eps)))

def ratio_mean(window):
    g1 = window[:-1]
    g2 = window[1:]
    r = np.minimum(g1, g2) / np.maximum(g1, g2)
    return float(np.mean(r))

def ratio_std(window):
    g1 = window[:-1]
    g2 = window[1:]
    r = np.minimum(g1, g2) / np.maximum(g1, g2)
    return float(np.std(r))

def build_windows_and_features(gaps, feature_family="minimal", k=5):
    windows = extract_windows(gaps, k=k)
    windows_n = np.array([normalize_window(w) for w in windows])

    stats = {
        "entropy": np.array([window_entropy(w) for w in windows_n]),
        "unevenness": np.array([unevenness(w) for w in windows_n]),
        "ratio_mean": np.array([ratio_mean(w) for w in windows_n]),
        "symmetry": np.array([reversal_symmetry_score(w) for w in windows_n]),
    }

    if feature_family == "minimal":
        X = np.array([
            [stats["entropy"][i], stats["unevenness"][i], stats["ratio_mean"][i]]
            for i in range(len(windows_n))
        ], dtype=float)
    elif feature_family == "full":
        X = np.array([
            [stats["entropy"][i], stats["unevenness"][i], stats["symmetry"][i], stats["ratio_mean"][i], ratio_std(windows_n[i])]
            for i in range(len(windows_n))
        ], dtype=float)
    elif feature_family == "raw_window":
        X = windows_n.copy()
    else:
        raise ValueError(f"Unknown feature family: {feature_family}")

    return windows_n, stats, X

def apply_condition_mask(stats, condition_name):
    if condition_name == "all":
        return np.ones(len(stats["entropy"]), dtype=bool)
    if condition_name == "low_entropy":
        thr = np.median(stats["entropy"])
        return stats["entropy"] <= thr
    if condition_name == "high_entropy":
        thr = np.median(stats["entropy"])
        return stats["entropy"] > thr
    if condition_name == "low_unevenness":
        thr = np.median(stats["unevenness"])
        return stats["unevenness"] <= thr
    if condition_name == "high_unevenness":
        thr = np.median(stats["unevenness"])
        return stats["unevenness"] > thr
    raise ValueError(condition_name)

## RBF overlap evaluator

In [ ]:
def split_train_test(X, frac=0.6):
    n = len(X)
    cut = int(frac * n)
    return X[:cut], X[cut:]

def standardize_train_test(X_train, X_test):
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0)
    std[std == 0] = 1.0
    return (X_train - mean) / std, (X_test - mean) / std

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -40, 40)))

def fit_logistic_regression(X, y, lr=0.1, n_steps=2500, reg=1e-4):
    Xb = np.column_stack([np.ones(len(X)), X])
    w = np.zeros(Xb.shape[1])
    for _ in range(n_steps):
        p = sigmoid(Xb @ w)
        grad = Xb.T @ (p - y) / len(X)
        grad[1:] += reg * w[1:]
        w -= lr * grad
    return w

def decision_function_logistic(X, w):
    Xb = np.column_stack([np.ones(len(X)), X])
    return Xb @ w

def predict_proba_logistic(X, w):
    return sigmoid(decision_function_logistic(X, w))

def choose_prototypes(X, n_proto=20):
    idx = np.linspace(0, len(X) - 1, n_proto).astype(int)
    return X[idx]

def estimate_rbf_gamma(X):
    D = np.linalg.norm(X[:, None, :] - X[None, :, :], axis=2)
    med = np.median(D[D > 0])
    if med <= 0:
        med = 1.0
    return 1.0 / (2.0 * med * med)

def rbf_features(X, prototypes, gamma):
    D2 = np.sum((X[:, None, :] - prototypes[None, :, :])**2, axis=2)
    return np.exp(-gamma * D2)

def accuracy(y_true, y_pred):
    return float(np.mean(y_true == y_pred))

def overlap_coefficient_from_hist(a, b, bins=30):
    lo = min(a.min(), b.min())
    hi = max(a.max(), b.max())
    if hi <= lo:
        return 1.0
    hist_a, edges = np.histogram(a, bins=bins, range=(lo, hi), density=True)
    hist_b, _ = np.histogram(b, bins=bins, range=(lo, hi), density=True)
    dx = edges[1] - edges[0]
    return float(np.sum(np.minimum(hist_a, hist_b)) * dx)

def evaluate_pair_rbf(X0, X1):
    m = min(len(X0), len(X1))
    X0 = X0[:m]
    X1 = X1[:m]

    X0_train, X0_test = split_train_test(X0, frac=0.6)
    X1_train, X1_test = split_train_test(X1, frac=0.6)

    X_train = np.vstack([X0_train, X1_train])
    y_train = np.array([0] * len(X0_train) + [1] * len(X1_train))
    X_test = np.vstack([X0_test, X1_test])
    y_test = np.array([0] * len(X0_test) + [1] * len(X1_test))

    Xtr_std, Xte_std = standardize_train_test(X_train, X_test)
    prototypes = choose_prototypes(Xtr_std, n_proto=min(20, len(Xtr_std)))
    gamma = estimate_rbf_gamma(Xtr_std)
    R_train = rbf_features(Xtr_std, prototypes, gamma)
    R_test = rbf_features(Xte_std, prototypes, gamma)

    w = fit_logistic_regression(R_train, y_train, lr=0.15, n_steps=3500, reg=1e-4)
    scores = decision_function_logistic(R_test, w)
    preds = (predict_proba_logistic(R_test, w) >= 0.5).astype(int)

    acc = accuracy(y_test, preds)
    scores0 = scores[y_test == 0]
    scores1 = scores[y_test == 1]

    return {
        "accuracy": acc,
        "score_overlap": overlap_coefficient_from_hist(scores0, scores1, bins=30),
    }

## Bootstrap helper

In [ ]:
def bootstrap_rows(X, rng):
    idx = rng.integers(0, len(X), size=len(X))
    return X[idx]

def bootstrap_pair_rbf(X0, X1, n_boot=30, seed=9423):
    boot_rng = np.random.default_rng(seed)
    acc_vals = []
    score_vals = []

    m = min(len(X0), len(X1))
    X0 = X0[:m]
    X1 = X1[:m]

    for _ in range(n_boot):
        X0b = bootstrap_rows(X0, boot_rng)
        X1b = bootstrap_rows(X1, boot_rng)
        out = evaluate_pair_rbf(X0b, X1b)
        acc_vals.append(out["accuracy"])
        score_vals.append(out["score_overlap"])

    acc_vals = np.array(acc_vals)
    score_vals = np.array(score_vals)

    return {
        "acc_mean": float(acc_vals.mean()),
        "acc_std": float(acc_vals.std()),
        "acc_q025": float(np.quantile(acc_vals, 0.025)),
        "acc_q975": float(np.quantile(acc_vals, 0.975)),
        "score_mean": float(score_vals.mean()),
        "score_std": float(score_vals.std()),
        "score_q025": float(np.quantile(score_vals, 0.025)),
        "score_q975": float(np.quantile(score_vals, 0.975)),
    }

## Surrogate builders

In [ ]:
def surrogate_baseline(gaps, rng=None):
    return np.array(gaps, dtype=float).copy()

def surrogate_distribution_only(gaps, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    gaps = np.array(gaps, dtype=float)
    return gaps[rng.permutation(len(gaps))]

def surrogate_iid_histogram(gaps, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    gaps = np.array(gaps, dtype=float)
    idx = rng.integers(0, len(gaps), size=len(gaps))
    return gaps[idx]

def surrogate_local_blocks(gaps, block_size=10, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    gaps = np.array(gaps, dtype=float)
    blocks = [gaps[i:i+block_size] for i in range(0, len(gaps), block_size)]
    perm = rng.permutation(len(blocks))
    return np.concatenate([blocks[i] for i in perm])

def surrogate_order_only(gaps):
    gaps = np.array(gaps, dtype=float)
    order = np.argsort(gaps)
    ranks = np.empty_like(order)
    ranks[order] = np.arange(len(gaps))
    q = np.linspace(gaps.min(), gaps.max(), len(gaps))
    return q[ranks]

def surrogate_global_trend(gaps, smooth_window=21):
    gaps = np.array(gaps, dtype=float)
    if smooth_window % 2 == 0:
        smooth_window += 1
    pad = smooth_window // 2
    padded = np.pad(gaps, pad, mode="edge")
    kernel = np.ones(smooth_window) / smooth_window
    smooth = np.convolve(padded, kernel, mode="valid")
    return np.clip(smooth, 1e-8, None)

## Reduced experiment grid

In [ ]:
window_sizes = [3, 5, 7]
feature_family = "minimal"
sample_size = 160
height_block = (0, 400)
n_boot = 30

conditions = ["low_entropy", "high_entropy", "low_unevenness", "high_unevenness"]

decomposition_grid = [
    ("baseline", [0]),
    ("distribution_only", [1]),
    ("iid_histogram", [1]),
    ("local_blocks", [10]),
    ("order_only", [1]),
    ("global_trend", [21]),
]

## Base gap slices

In [ ]:
start, stop = height_block
zeta_base_gaps = zeta_gaps_full[start:stop]
poisson_base_gaps = poisson_gaps_full[start:stop]
gue_base_gaps = gue_gaps_full[:max(stop - start + 300, 900)]
len(zeta_base_gaps), len(poisson_base_gaps), len(gue_base_gaps)

## Main conditional decomposition sweep

In [ ]:
cond_results = []

for k in window_sizes:
    _, zeta_base_stats, _ = build_windows_and_features(zeta_base_gaps, feature_family=feature_family, k=k)

    for condition_name in conditions:
        _, p_stats, pX_all = build_windows_and_features(poisson_base_gaps, feature_family=feature_family, k=k)
        p_mask = apply_condition_mask(p_stats, condition_name)

        _, g_stats, gX_all = build_windows_and_features(gue_base_gaps, feature_family=feature_family, k=k)
        g_mask = apply_condition_mask(g_stats, condition_name)

        pX_ref = pX_all[p_mask]
        gX_ref = gX_all[g_mask]

        for decomp_name, strengths in decomposition_grid:
            for strength in strengths:
                local_rng = np.random.default_rng(9423 + 100*k + 1000*len(condition_name) + int(1000*strength) + len(decomp_name))

                if decomp_name == "baseline":
                    zeta_sur = surrogate_baseline(zeta_base_gaps, rng=local_rng)
                elif decomp_name == "distribution_only":
                    zeta_sur = surrogate_distribution_only(zeta_base_gaps, rng=local_rng)
                elif decomp_name == "iid_histogram":
                    zeta_sur = surrogate_iid_histogram(zeta_base_gaps, rng=local_rng)
                elif decomp_name == "local_blocks":
                    zeta_sur = surrogate_local_blocks(zeta_base_gaps, block_size=int(strength), rng=local_rng)
                elif decomp_name == "order_only":
                    zeta_sur = surrogate_order_only(zeta_base_gaps)
                elif decomp_name == "global_trend":
                    zeta_sur = surrogate_global_trend(zeta_base_gaps, smooth_window=int(strength))
                else:
                    raise ValueError(decomp_name)

                _, z_stats, zX_all = build_windows_and_features(zeta_sur, feature_family=feature_family, k=k)
                z_mask = apply_condition_mask(z_stats, condition_name)
                zX_ref = zX_all[z_mask]

                m = min(len(zX_ref), len(pX_ref), len(gX_ref), sample_size)
                if m < 40:
                    continue

                zX = zX_ref[:m]
                pX = pX_ref[:m]
                gX = gX_ref[:m]

                out_zg = bootstrap_pair_rbf(zX, gX, n_boot=n_boot, seed=8000 + 10*k + int(1000*strength) + len(condition_name))
                out_zp = bootstrap_pair_rbf(zX, pX, n_boot=n_boot, seed=9000 + 10*k + int(1000*strength) + len(condition_name))

                cond_results.append({
                    "k": k,
                    "condition": condition_name,
                    "decomposition": decomp_name,
                    "strength": strength,
                    "sample_used": m,
                    "zg_score_mean": out_zg["score_mean"],
                    "zg_score_std": out_zg["score_std"],
                    "zg_score_q025": out_zg["score_q025"],
                    "zg_score_q975": out_zg["score_q975"],
                    "zp_score_mean": out_zp["score_mean"],
                    "zp_score_std": out_zp["score_std"],
                    "zp_score_q025": out_zp["score_q025"],
                    "zp_score_q975": out_zp["score_q975"],
                    "phase_gap_mean": out_zg["score_mean"] - out_zp["score_mean"],
                })

len(cond_results)

## Quick diagnostic

In [ ]:
sorted(set(r["condition"] for r in cond_results)), sorted(set(r["decomposition"] for r in cond_results)), sorted(set(r["k"] for r in cond_results))

## Conditional phase-gap bars

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharey=True)
axes = axes.ravel()

for ax, condition_name in zip(axes, conditions):
    rows = [r for r in cond_results if r["condition"] == condition_name and r["k"] == 5]
    labels = []
    vals = []
    for decomp_name, strengths in decomposition_grid:
        for s in strengths:
            match = [r for r in rows if r["decomposition"] == decomp_name and r["strength"] == s]
            if match:
                labels.append(f"{decomp_name}:{s}")
                vals.append(match[0]["phase_gap_mean"])
    ax.bar(range(len(vals)), vals)
    ax.set_xticks(range(len(vals)), labels, rotation=45, ha="right")
    ax.set_title(f"{condition_name} (k=5)")
    ax.set_ylabel("phase gap")

plt.tight_layout()
plt.show()

## Conditional phase-gap heatmaps

In [ ]:
selected_decomps = [("baseline", 0), ("distribution_only", 1), ("iid_histogram", 1), ("local_blocks", 10), ("order_only", 1), ("global_trend", 21)]

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey=True)
axes = axes.ravel()

for ax, condition_name in zip(axes, conditions):
    rows = []
    labels = []
    for decomp_name, s in selected_decomps:
        labels.append(f"{decomp_name}:{s}")
        row = []
        for k in window_sizes:
            match = next(r for r in cond_results if r["condition"] == condition_name and r["decomposition"] == decomp_name and r["strength"] == s and r["k"] == k)
            row.append(match["phase_gap_mean"])
        rows.append(row)
    M = np.array(rows)
    im = ax.imshow(M, aspect="auto", origin="upper")
    ax.set_title(condition_name)
    ax.set_xticks(np.arange(len(window_sizes)), window_sizes)
    ax.set_yticks(np.arange(len(labels)), labels)

fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.85, label="phase gap")
plt.tight_layout()
plt.show()

## Condition-difference heatmaps

In [ ]:
def condition_matrix(condition_name):
    rows = []
    for decomp_name, s in selected_decomps:
        row = []
        for k in window_sizes:
            match = next(r for r in cond_results if r["condition"] == condition_name and r["decomposition"] == decomp_name and r["strength"] == s and r["k"] == k)
            row.append(match["phase_gap_mean"])
        rows.append(row)
    return np.array(rows)

M_le = condition_matrix("low_entropy")
M_he = condition_matrix("high_entropy")
M_lu = condition_matrix("low_unevenness")
M_hu = condition_matrix("high_unevenness")

diff_entropy = M_he - M_le
diff_unevenness = M_hu - M_lu

fig, axes = plt.subplots(1, 2, figsize=(10.5, 6), sharey=True)

for ax, M, title in zip(
    axes,
    [diff_entropy, diff_unevenness],
    ["high_entropy minus low_entropy", "high_unevenness minus low_unevenness"]
):
    im = ax.imshow(M, aspect="auto", origin="upper")
    ax.set_title(title)
    ax.set_xticks(np.arange(len(window_sizes)), window_sizes)
    ax.set_yticks(np.arange(len(selected_decomps)), [f"{d}:{s}" for d, s in selected_decomps])

fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.85, label="difference in phase gap")
plt.tight_layout()
plt.show()

## Bootstrap error-bar comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=False)

compare_conditions = ["low_entropy", "high_entropy"]
compare_decomps = [("baseline", 0), ("iid_histogram", 1), ("local_blocks", 10), ("global_trend", 21)]

for ax, condition_name in zip(axes, compare_conditions):
    for decomp_name, s in compare_decomps:
        xs = []
        means = []
        lows = []
        highs = []
        for k in window_sizes:
            match = next(r for r in cond_results if r["condition"] == condition_name and r["decomposition"] == decomp_name and r["strength"] == s and r["k"] == k)
            xs.append(k)
            means.append(match["phase_gap_mean"])
            low = match["zg_score_mean"] - match["zg_score_q025"]
            high = match["zg_score_q975"] - match["zg_score_mean"]
            lows.append(max(0.0, low))
            highs.append(max(0.0, high))
        ax.errorbar(xs, means, yerr=np.vstack([np.array(lows), np.array(highs)]), marker="o", capsize=4, label=f"{decomp_name}:{s}")
    ax.set_title(condition_name)
    ax.set_xlabel("window size k")
    ax.set_ylabel("phase gap")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## Compact condition summary

In [ ]:
summary_rows = []
for condition_name in conditions:
    vals = [r["phase_gap_mean"] for r in cond_results if r["condition"] == condition_name]
    summary_rows.append({
        "condition": condition_name,
        "mean_phase_gap": float(np.mean(vals)),
        "min_phase_gap": float(np.min(vals)),
        "max_phase_gap": float(np.max(vals)),
    })
summary_rows

## Compact summary

In [ ]:
summary = {
    "window_sizes": window_sizes,
    "feature_family": feature_family,
    "sample_size_target": sample_size,
    "height_block": f"{height_block[0]+1}-{height_block[1]}",
    "n_boot": n_boot,
    "conditions": conditions,
    "selected_decompositions": [f"{d}:{s}" for d, s in selected_decomps],
    "condition_summary": summary_rows,
}
summary

## Notes

- If one condition has consistently larger phase gaps across decomposition families, then the overlap phase is concentrated in that local regime.
- If iid-histogram and local-block surrogates stay strong only in one condition, that suggests the mechanism is not uniform across all windows.
- This notebook uses median splits for interpretability and stable sample sizes.

## Next directions

A natural next notebook could do one of these:

1. cross-condition transfer: train on low-entropy, test on high-entropy, and vice versa  
2. repeat the same analysis for additional feature families  
3. run the conditional decomposition on conditional-window / sequential features  
4. compare conditional confidence geometry rather than overlap alone